In [1]:
!pip install torch torchvision

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [3]:
train_data = datasets.MNIST(root="./data",train=True,download=True,transform=transform)
test_data = datasets.MNIST(root="./data",train=False,download=True,transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64,shuffle=False)


In [4]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cpu'

In [6]:
model = MNISTCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [7]:
def train(model, loader):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [8]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


In [ ]:
epochs = 5
train_losses = []
test_accuracies = []

for epoch in range(epochs):
    loss = train(model, train_loader)
    acc = evaluate(model, test_loader)

    train_losses.append(loss)
    test_accuracies.append(acc)

    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss:.4f} | Accuracy: {acc:.4f}")


Epoch 1/5 | Loss: 0.1160 | Accuracy: 0.9845


In [ ]:
import os
os.makedirs("mnist_app/models", exist_ok=True)
torch.save(model.state_dict(), "mnist_app/models/mnist_cnn.pth")
print("Model saved!")


In [ ]:
def predict(image, model):
    model.eval()
    image = image.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image)
        probs = F.softmax(output, dim=1)
        pred = probs.argmax(dim=1).item()
        conf = probs[0][pred].item()

    return pred, conf


In [ ]:
img, label = test_data[0]
pred, conf = predict(img, model)

plt.imshow(img.squeeze(), cmap="gray")
plt.title(f"Predicted: {pred} | Confidence: {conf:.2f}")
plt.axis("off")


Plots for submission

In [ ]:
plt.plot(train_losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

Accuracy Plot

In [ ]:
plt.plot(test_accuracies)
plt.title("Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()